# 05 — Modelagem dimensional e integração da Tríade

**Projeto:** Plataforma Financeira Integrada — Tríade (Linha 4)

**Entrada:** tabelas tratadas de `data/processed/` (`fato_macro`, `fato_cotacoes`, `fato_contabil`).

**Modelo:** esquema estrela simplificado — uma dimensão de empresas (`dim_empresa`) ligada às tabelas de fatos, e uma tabela integrada anual (`fato_anual_integrado`) com uma linha por empresa e exercício.

**Saída:** `dim_empresa.csv`, `fato_indicadores_contabeis.csv`, `fato_macro_anual.csv`, `fato_mercado_anual.csv` e `fato_anual_integrado.csv` em `data/processed/`.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 100)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"

macro = pd.read_csv(PROC_DIR / "fato_macro.csv", parse_dates=["data"])
cotacoes = pd.read_csv(PROC_DIR / "fato_cotacoes.csv", parse_dates=["data"])
contabil = pd.read_csv(PROC_DIR / "fato_contabil.csv", dtype={"cnpj": str, "cd_conta": str})

print("macro:", macro.shape, "| cotacoes:", cotacoes.shape, "| contabil:", contabil.shape)

macro: (3201, 3) | cotacoes: (11169, 9) | contabil: (9894, 8)


In [2]:
CARTEIRA = {
    "LREN3": ("Lojas Renner",  "Consumo/Varejo"),
    "ABEV3": ("Ambev",         "Consumo/Varejo"),
    "PETR4": ("Petrobras",     "Energia"),
    "EGIE3": ("Engie Brasil",  "Energia"),
    "VALE3": ("Vale",          "Commodities Exportadoras"),
    "SUZB3": ("Suzano",        "Commodities Exportadoras"),
    "WEGE3": ("WEG",           "Exportadoras Industriais"),
    "EMBR3": ("Embraer",       "Exportadoras Industriais"),
    "ITUB4": ("Itaú Unibanco", "Bancos"),
}

dim_empresa = pd.DataFrame(
    [(ticker, nome, setor) for ticker, (nome, setor) in CARTEIRA.items()],
    columns=["ticker", "empresa", "setor"],
)
dim_empresa["tipo"] = dim_empresa["setor"].map(lambda s: "Financeira" if s == "Bancos" else "Não financeira")
dim_empresa = dim_empresa.merge(contabil[["ticker", "cnpj"]].drop_duplicates(), on="ticker", how="left")

dim_empresa.to_csv(PROC_DIR / "dim_empresa.csv", index=False)
dim_empresa

,ticker,empresa,setor,tipo,cnpj
0,LREN3,Lojas Renner,Consumo/Varejo,Não financeira,92.754.738/0001-62
1,ABEV3,Ambev,Consumo/Varejo,Não financeira,07.526.557/0001-00
2,PETR4,Petrobras,Energia,Não financeira,33.000.167/0001-01
3,EGIE3,Engie Brasil,Energia,Não financeira,02.474.103/0001-19
4,VALE3,Vale,Commodities Exportadoras,Não financeira,33.592.510/0001-54
5,SUZB3,Suzano,Commodities Exportadoras,Não financeira,16.404.287/0001-55
6,WEGE3,WEG,Exportadoras Industriais,Não financeira,84.429.695/0001-11
7,EMBR3,Embraer,Exportadoras Industriais,Não financeira,07.689.002/0001-89
8,ITUB4,Itaú Unibanco,Bancos,Financeira,60.872.504/0001-23


In [3]:
CONTAS = {
    "receita_liquida": "3.01",
    "resultado_bruto": "3.03",
    "ebit": "3.05",
    "despesas_financeiras": "3.06.02",
    "lucro_liquido": "3.11",
    "ativo_total": "1",
    "ativo_circulante": "1.01",
    "caixa": "1.01.01",
    "aplicacoes_financeiras": "1.01.02",
    "passivo_circulante": "2.01",
    "emprestimos_cp": "2.01.04",
    "emprestimos_lp": "2.02.01",
    "patrimonio_liquido": "2.03",
}

nao_fin = dim_empresa.loc[dim_empresa["tipo"] == "Não financeira", "ticker"]
recorte = contabil[(contabil["ano"] == 2023)
                   & contabil["ticker"].isin(nao_fin)
                   & contabil["cd_conta"].isin(CONTAS.values())]

(recorte.groupby(["cd_conta", "ds_conta"])["ticker"]
        .apply(lambda s: ", ".join(sorted(s)))
        .reset_index(name="empresas"))

,cd_conta,ds_conta,empresas
0,1,Ativo Total,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
1,1.01,Ativo Circulante,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
2,1.01.01,Caixa e Equivalentes de Caixa,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
3,1.01.02,Aplicações Financeiras,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
4,2.01,Passivo Circulante,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
5,2.01.04,Empréstimos e Financiamentos,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
6,2.02.01,Empréstimos e Financiamentos,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
7,2.03,Patrimônio Líquido Consolidado,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
8,3.01,Receita de Venda de Bens e/ou Serviços,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."
9,3.03,Resultado Bruto,"ABEV3, EGIE3, EMBR3, LREN3, PETR4, SUZB3, VALE..."


In [4]:
itau = contabil[(contabil["ticker"] == "ITUB4")
                & (contabil["ano"] == 2023)
                & (contabil["cd_conta"].str.count(r"\.") <= 1)]
itau[["demonstrativo", "cd_conta", "ds_conta", "valor_reais"]]

,demonstrativo,cd_conta,ds_conta,valor_reais
4195,BPA,1,Ativo Total,2.543100e+12
4196,BPA,1.01,Caixa e Equivalentes de Caixa,3.200100e+10
4197,BPA,1.02,Ativos Financeiros,2.384618e+12
4216,BPA,1.03,Tributos Diferidos,6.452100e+10
4220,BPA,1.04,Outros Ativos,2.016800e+10
4225,BPA,1.05,Investimentos,9.293000e+09
4230,BPA,1.06,Imobilizado,9.135000e+09
4233,BPA,1.07,Intangível,2.336400e+10
4236,BPP,2,Passivo Total,2.543100e+12
4237,BPP,2.01,Passivos Financeiros ao Valor Justo através do...,5.333100e+10


## Etapa 1 — Conclusão: dimensão e plano de contas

- `dim_empresa`: 9 companhias, 5 setores, classificadas em **Não financeira** (8) e **Financeira** (Itaú).
- Os 13 códigos contábeis usados nas empresas não financeiras têm **descrição única e idêntica** em todas elas (plano de contas padronizado da CVM).
- O **Itaú** segue o plano de contas de instituição financeira: Lucro Líquido = **3.09**, Patrimônio Líquido = **2.08**, Receita = **3.01** (intermediação financeira). Não há EBIT, Ativo/Passivo Circulante nem Dívida Líquida comparáveis.
- **Decisão:** para o Itaú são calculados apenas Receita, Lucro, PL e ROE; os demais indicadores ficam como não aplicáveis (N/A).

In [5]:
CONTAS_BANCO = {
    "receita_liquida": "3.01",    # Receitas da Intermediação Financeira
    "lucro_liquido": "3.09",      # Lucro/Prejuízo Consolidado do Período
    "ativo_total": "1",
    "patrimonio_liquido": "2.08",
}

(contabil[(contabil["ticker"] == "ITUB4") & contabil["cd_conta"].isin(CONTAS_BANCO.values())]
    .pivot_table(index="cd_conta", columns="ano", values="ds_conta", aggfunc="first"))

ano,2019,2020,2021,2022,2023
cd_conta,,,,,
1,Ativo Total,Ativo Total,Ativo Total,Ativo Total,Ativo Total
2.08,Patrimônio Líquido Consolidado,Patrimônio Líquido Consolidado,Patrimônio Líquido Consolidado,Patrimônio Líquido Consolidado,Patrimônio Líquido Consolidado
3.01,Receitas da Intermediação Financeira,Receitas da Intermediação Financeira,Receitas da Intermediação Financeira,Receitas da Intermediação Financeira,Receitas da Intermediação Financeira
3.09,Lucro/Prejuízo Consolidado do Período,Lucro/Prejuízo Consolidado do Período,Lucro/Prejuízo Consolidado do Período,Lucro/Prejuízo Consolidado do Período,Lucro/Prejuízo Consolidado do Período


In [6]:
def extrair_contas(df, mapa):
    """Seleciona as contas do mapa e organiza: uma linha por empresa/ano, uma coluna por conta."""
    codigo_para_nome = {codigo: nome for nome, codigo in mapa.items()}
    selecao = df[df["cd_conta"].isin(codigo_para_nome)].copy()
    selecao["conta"] = selecao["cd_conta"].map(codigo_para_nome)
    return (selecao.pivot_table(index=["ticker", "ano"], columns="conta",
                                values="valor_reais", aggfunc="first")
                   .reset_index())


ind_nao_fin = extrair_contas(contabil[contabil["ticker"].isin(nao_fin)], CONTAS)
ind_banco = extrair_contas(contabil[contabil["ticker"] == "ITUB4"], CONTAS_BANCO)

# Completude: contas faltantes em cada grupo (o esperado é zero)
print("Não financeiras — contas faltantes por coluna:")
print(ind_nao_fin.isna().sum())
print("\nItaú — contas faltantes por coluna:")
print(ind_banco.isna().sum())

Não financeiras — contas faltantes por coluna:
conta
ticker                    0
ano                       0
aplicacoes_financeiras    0
ativo_circulante          0
ativo_total               0
caixa                     0
despesas_financeiras      0
ebit                      0
emprestimos_cp            0
emprestimos_lp            0
lucro_liquido             0
passivo_circulante        0
patrimonio_liquido        0
receita_liquida           0
resultado_bruto           0
dtype: int64

Itaú — contas faltantes por coluna:
conta
ticker                0
ano                   0
ativo_total           0
lucro_liquido         0
patrimonio_liquido    0
receita_liquida       0
dtype: int64


In [9]:
ind = pd.concat([ind_nao_fin, ind_banco], ignore_index=True)
ind.columns.name = None  # remove o rótulo "conta" herdado do pivot

# Dívida líquida = dívida bruta (curto + longo prazo) − caixa e aplicações de curto prazo
ind["divida_bruta"] = ind["emprestimos_cp"] + ind["emprestimos_lp"]
ind["caixa_total"] = ind["caixa"] + ind["aplicacoes_financeiras"].fillna(0)
ind["divida_liquida"] = ind["divida_bruta"] - ind["caixa_total"]

# Indicadores (em fração: 0,15 = 15%)
ind["margem_bruta"] = ind["resultado_bruto"] / ind["receita_liquida"]
ind["margem_ebit"] = ind["ebit"] / ind["receita_liquida"]
ind["margem_liquida"] = ind["lucro_liquido"] / ind["receita_liquida"]
ind["roe"] = ind["lucro_liquido"] / ind["patrimonio_liquido"]
ind["liquidez_corrente"] = ind["ativo_circulante"] / ind["passivo_circulante"]
ind["divida_liquida_pl"] = ind["divida_liquida"] / ind["patrimonio_liquido"]

ind = ind.sort_values(["ticker", "ano"]).reset_index(drop=True)
ind["crescimento_receita"] = ind.groupby("ticker")["receita_liquida"].pct_change()

# Banco: margem líquida sobre receita de intermediação não é comparável às demais empresas
ind.loc[ind["ticker"] == "ITUB4", "margem_liquida"] = float("nan")

ind.to_csv(PROC_DIR / "fato_indicadores_contabeis.csv", index=False)
print("Linhas:", len(ind))

Linhas: 45


In [10]:
validacao = ind.loc[ind["ano"] == 2023,
                    ["ticker", "receita_liquida", "lucro_liquido", "roe", "margem_ebit", "liquidez_corrente"]].copy()
validacao[["receita_liquida", "lucro_liquido"]] /= 1e9  # em R$ bilhões
validacao.round(2)

,ticker,receita_liquida,lucro_liquido,roe,margem_ebit,liquidez_corrente
4,ABEV3,79.74,14.96,0.19,0.23,0.89
9,EGIE3,10.75,3.43,0.35,0.59,1.46
14,EMBR3,26.11,0.78,0.05,0.06,1.65
19,ITUB4,313.22,33.88,0.17,NaN,NaN
24,LREN3,13.65,0.98,0.10,0.07,1.63
29,PETR4,511.99,125.17,0.33,0.37,0.96
34,SUZB3,39.76,14.11,0.31,0.31,2.61
39,VALE3,208.07,40.55,0.20,0.31,1.28
44,WEGE3,32.50,5.87,0.33,0.20,1.92


## Etapa 2 — Conclusão: indicadores contábeis

- **45 registros** (9 empresas × 5 exercícios), sem contas faltantes; códigos do Itaú estáveis de 2019 a 2023.
- **Validação de ordem de grandeza (2023):** receitas e lucros compatíveis com os resultados divulgados pelas companhias (ex.: Petrobras ≈ R$ 125 bi de lucro; WEG ≈ R$ 32,5 bi de receita).
- **Indicadores em fração** (0,15 = 15%): margem bruta, margem EBIT, margem líquida, ROE, liquidez corrente, dívida líquida/PL e crescimento da receita.
- **Simplificações declaradas:** ROE sobre o PL do fim do exercício; alavancagem medida por Dívida Líquida/PL (o EBITDA exigiria a depreciação, ausente da DRE); lucro consolidado (inclui minoritários).
- **Itaú:** margens, liquidez e dívida líquida não aplicáveis (N/A).